# BNP Paribas Cardif Claims Management


<img src='https://turkonfed.org/Files/Content/img_gelismekte-olan-sigorta-sektoru_96819.jpg_cropped.jpg'>


Bu çalışmada `BNP Paribas Cardif Claims Management` yarışması için müşteri ve talep özellikleri kullanılarak talep sonucu tahmini yapan bir başlangıç modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Veri dosyalarını yükleme
4. Ön işleme
5. Özellik çıkarımı
6. Model kurma
7. ROC AUC değerlendirmesi
8. Test tahmini ve submission
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
!pip install catboost -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import io
import os
import zipfile

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [3]:
# Bu bölümde yarışma zip dosyasını Drive üzerinden bağlıyor ve gerekli dosyaları buluyoruz.


In [4]:
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/bnp-paribas-cardif-claims-management.zip'
if not os.path.exists(zip_path):
    raise FileNotFoundError('Zip dosyası bulunamadı. Colab Data Dosyaları içindeki gerçek dosya adını kontrol et.')

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()

def find_zip_member(members, candidates):
    for candidate in candidates:
        for member in members:
            if member.endswith(candidate):
                return member
    raise FileNotFoundError(f'{candidates[0]} zip içinde bulunamadi.')

train_member = find_zip_member(zip_members, ['train.csv', 'train.csv.zip'])
test_member = find_zip_member(zip_members, ['test.csv', 'test.csv.zip'])
sample_submission_member = find_zip_member(zip_members, ['sample_submission.csv', 'sample_submission.csv.zip'])


Mounted at /content/drive


## 3. Veri Dosyalarını Yükleme


In [5]:
# Bu bölümde yarışma dosyalarını doğrudan zip içinden yüklüyoruz.


In [6]:
def read_csv_from_zip(zip_path, member_name):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(member_name) as f:
            if member_name.endswith('.zip'):
                inner_bytes = f.read()
                with zipfile.ZipFile(io.BytesIO(inner_bytes), 'r') as inner_zip:
                    inner_name = inner_zip.namelist()[0]
                    with inner_zip.open(inner_name) as inner_f:
                        return pd.read_csv(inner_f)
            return pd.read_csv(f)

train = read_csv_from_zip(zip_path, train_member)
test = read_csv_from_zip(zip_path, test_member)
sample_submission = read_csv_from_zip(zip_path, sample_submission_member)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Sample submission shape:', sample_submission.shape)
train.head()


Train shape: (114321, 133)
Test shape: (114393, 132)
Sample submission shape: (114393, 2)


,ID,target,v1,v2,v3,v4,v5,v6,v7,v8,v9,v10,v11,v12,v13,v14,v15,v16,v17,v18,v19,v20,v21,v22,v23,v24,v25,v26,v27,v28,v29,v30,v31,v32,v33,v34,v35,v36,v37,v38,v39,v40,v41,v42,v43,v44,v45,v46,v47,v48,v49,v50,v51,v52,v53,v54,v55,v56,v57,v58,v59,v60,v61,v62,v63,v64,v65,v66,v67,v68,v69,v70,v71,v72,v73,v74,v75,v76,v77,v78,v79,v80,v81,v82,v83,v84,v85,v86,v87,v88,v89,v90,v91,v92,v93,v94,v95,v96,v97,v98,v99,v100,v101,v102,v103,v104,v105,v106,v107,v108,v109,v110,v111,v112,v113,v114,v115,v116,v117,v118,v119,v120,v121,v122,v123,v124,v125,v126,v127,v128,v129,v130,v131
0,3,1,1.335739,8.727474,C,3.921026,7.915266,2.599278,3.176895,0.012941,9.999999,0.503281,16.434108,6.085711,2.866830,11.636387,1.355013,8.571429,3.670350,0.106720,0.148883,18.869283,7.730923,XDX,-1.716131e-08,C,0.139412,1.720818,3.393503,0.590122,8.880867,C,A,1.083033,1.010829,7.270147,8.375452,11.326592,0.454546,0,4.012088,7.711453,7.653429,12.707581,2.015505,10.498338,9.848672,0.113561,C,12.171733,8.086643,0.899420,7.277792,G,16.747968,0.037096,1.299638,DI,3.971118,0.529802,10.890984,1.588448,15.858152,1,0.153461,6.363189,18.303925,C,9.314079,15.231789,17.142857,11.784549,F,1,1.614988,B,D,2.230940,7.292418,8.571429,E,3.000000,7.528326,8.861647,0.649820,1.299638,1.707317,0.866426,9.551836,3.321300,0.095678,0.905342,A,0.442252,5.814018,3.517720,0.462019,7.436824,5.454545,8.877414,1.191337,19.470199,8.389237,2.757375,4.374296,1.574039,0.007294,12.579184,E,2.382692,3.930922,B,0.433213,O,NaN,15.634907,2.857144,1.951220,6.592012,5.909091,-6.297423e-07,1.059603,0.803572,8.000000,1.989780,0.035754,AU,1.804126,3.113719,2.024285,0,0.636365,2.857144
1,4,1,NaN,NaN,C,NaN,9.191265,NaN,NaN,2.301630,NaN,1.312910,NaN,6.507647,NaN,11.636386,NaN,NaN,NaN,NaN,NaN,NaN,6.763110,GUV,NaN,C,3.056144,NaN,NaN,NaN,NaN,C,A,NaN,NaN,3.615077,NaN,14.579479,NaN,0,NaN,14.305766,NaN,NaN,NaN,NaN,NaN,2.449959,E,NaN,NaN,1.379210,NaN,G,NaN,1.129469,NaN,DY,NaN,NaN,NaN,NaN,NaN,2,2.544736,NaN,NaN,A,NaN,NaN,NaN,12.053353,F,2,NaN,B,D,NaN,NaN,NaN,D,NaN,7.277655,3.430691,NaN,NaN,NaN,NaN,9.848004,NaN,2.678584,NaN,B,NaN,NaN,NaN,NaN,NaN,NaN,8.303967,NaN,NaN,NaN,NaN,NaN,NaN,1.505335,NaN,B,1.825361,4.247858,A,NaN,U,G,10.308044,NaN,NaN,10.595357,NaN,NaN,NaN,NaN,NaN,NaN,0.598896,AF,NaN,NaN,1.957825,0,NaN,NaN
2,5,1,0.943877,5.310079,C,4.410969,5.326159,3.979592,3.928571,0.019645,12.666667,0.765864,14.756098,6.384670,2.505589,9.603542,1.984127,5.882353,3.170847,0.244541,0.144258,17.952332,5.245035,FQ,-2.785053e-07,E,0.113997,2.244897,5.306122,0.836005,7.499999,NaN,A,1.454082,1.734693,4.043864,7.959184,12.730517,0.259740,0,7.378964,13.077201,6.173469,12.346939,2.926830,8.897561,5.343819,0.126035,C,12.711328,6.836734,0.604504,9.637627,F,15.102041,0.085573,0.765305,AS,4.030613,4.277456,9.105481,2.151361,16.075602,1,0.123643,5.517949,16.377205,A,8.367347,11.040463,5.882353,8.460654,B,3,2.413618,B,B,1.963971,5.918368,11.764705,E,3.333334,10.194433,8.266200,1.530611,1.530613,2.429906,1.071429,8.447465,3.367346,0.111388,0.811447,G,0.271480,5.156559,4.214944,0.309657,5.663265,5.974026,11.588858,0.841837,15.491329,5.879353,3.292788,5.924457,1.668401,0.008275,11.670572,C,1.375753,1.184211,B,3.367348,S,NaN,11.205561,12.941177,3.129253,3.478911,6.233767,-2.792745e-07,2.138728,2.238806,9.333333,2.477596,0.013452,AE,1.773709,3.922193,1.120468,2,0.883118,1.176472
3,6,1,0.797415,8.304757,C,4.225930,11.627438,2.097700,1.987549,0.171947,8.965516,6.542669,16.347483,9.646653,3.903302,14.094723,1.945044,5.517242,3.610789,1.224114,0.231630,18.376407,7.517125,ACUE,-4.805344e-07,D,0.148843,1.308269,2.303640,8.926662,8.874521,C,B,1.587644,1.666667,8.703550,8.898468,11.302795,0.433735,0,0.287322,11.523045,7.931035,12.935823,1.470878,12.708574,9.670823,0.108387,C,12.194855,8.591954,3.329176,4.780357,H,16.621695,0.139721,1.178161,BW,3.965517,1.732102,11.777912,1.229246,15.927390,1,0.140260,6.292979,17.011645,A,9.703065,18.568129,9.425288,13.594728,F,2,2.272541,B,D,2.188198,8.213602,13.448277,B,1.947261,4.797873,13.315819,1.681034,1.379310,1.587045,1.242817,10.

## 4. Ön İşleme


In [7]:
# Bu bölümde hedef değişkeni ayırıyor ve temel veri temizliğini yapıyoruz.


In [8]:
target_col = 'target'

train = train.replace([np.inf, -np.inf], np.nan)
test = test.replace([np.inf, -np.inf], np.nan)

feature_cols = [col for col in train.columns if col not in ['ID', target_col] and col in test.columns]


## 5. Özellik Çıkarımı


In [9]:
# Bu bölümde kategorik ve sayısal özellikleri modele uygun hale getiriyoruz.


In [10]:
train_x = train[feature_cols].copy()
train_y = train[target_col].copy()
test_x = test[feature_cols].copy()

categorical_cols = [col for col in feature_cols if train_x[col].dtype == 'object']
for col in categorical_cols:
    train_x[col] = train_x[col].astype(str)
    test_x[col] = test_x[col].astype(str)

numeric_cols = [col for col in feature_cols if col not in categorical_cols]
for col in numeric_cols:
    train_x[col] = pd.to_numeric(train_x[col], errors='coerce')
    test_x[col] = pd.to_numeric(test_x[col], errors='coerce')

x_train, x_valid, y_train, y_valid = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42, stratify=train_y
)

print('x_train shape:', x_train.shape)
print('x_valid shape:', x_valid.shape)


x_train shape: (91456, 131)
x_valid shape: (22865, 131)


## 6. Model Kurma


In [11]:
# Bu bölümde talep sonucu tahmini için CatBoost modelini kuruyoruz.


In [12]:
model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.08,
    depth=8,
    loss_function='Logloss',
    eval_metric='AUC',
    verbose=0,
    random_seed=42
)

model.fit(
    x_train,
    y_train,
    cat_features=categorical_cols,
    eval_set=(x_valid, y_valid),
    use_best_model=True
)


CatBoostClassifier(depth=8, eval_metric='AUC', iterations=300, learning_rate=0.08, loss_function='Logloss', random_seed=42, verbose=0)

## 7. ROC AUC Değerlendirmesi


In [13]:
# Bu bölümde validation verisi üzerindeki ROC AUC değerini hesaplıyoruz.


In [14]:
valid_probs = model.predict_proba(x_valid)[:, 1]
roc_auc = roc_auc_score(y_valid, valid_probs)
print('Validation ROC AUC:', round(roc_auc, 5))


Validation ROC AUC: 0.77585


## 8. Test Tahmini ve Submission


In [15]:
# Bu bölümde test tahminlerini submission formatına dönüştürüyoruz.


In [16]:
test_probs = model.predict_proba(test_x)[:, 1]
submission = sample_submission.copy()
if 'PredictedProb' in submission.columns:
    submission['PredictedProb'] = test_probs[:len(submission)]
else:
    submission.iloc[:, 1] = test_probs[:len(submission)]

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (114393, 2)


,ID,PredictedProb
0,0,0.197836
1,1,0.898626
2,2,0.852853
3,7,0.653268
4,10,0.810187


In [17]:
output_path = '/content/bnp_paribas_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/bnp_paribas_submission.csv


## 9. Sonuç


Bu çalışmada BNP Paribas Cardif Claims Management yarışması için müşteri ve talep özellikleri kullanılarak talep sonucu tahmini yapan bir başlangıç modeli oluşturuldu. Elde edilen sonuçlara göre model validation verisi üzerinde 0.77585 ROC AUC değeri elde etti ve test verisi için talep olasılıkları üretildi.
